# 06 — Generalization Eval: Base Gemma vs SFT Gemma · Google Colab

**Phase 2 / Month 2** · MSc Thesis — ECLIPSE project
Supervisor: Dr. Panagiotis Kasnesis | Student: Antonios Bastoulis

---

## What this notebook does

Compares the **base Gemma** (zero-shot, no fine-tuning) against the **SFT Gemma** (LoRA-distilled from SAC, nb 04 + nb 05) on building / week combinations the SFT model **has not been trained on** — i.e. the held-out 2022-dataset buildings (indices 6–11). Both models share the same in-memory weights via PEFT adapter toggling, so only one base-model load is needed per Colab session.

For each config in the eval grid we run four policies:

1. **No-Control** — every battery action = 0.0. The reference baseline that the Challenge KPIs normalise against.
2. **RBC** — `BasicBatteryRBC`, fixed solar-window rule. A non-learning floor.
3. **Base Gemma** — adapter bypassed via `model.disable_adapter()`. CoT prompt (`make_minimal_prompt`).
4. **SFT Gemma** — adapter enabled. No-CoT prompt (`make_sft_prompt`, the one it was trained on).

KPIs come from `src.eval.evaluate / comparison_table / generalisation_gap` (CityLearn 2.6 `evaluate_v2`) so the numbers are directly comparable to nb 01 / nb 04 / nb 05.

## Why adapter toggling, not two separate model loads

Gemma-4 E4B in 4-bit is ~9.5 GB — two copies don't fit in a 16 GB T4 / 22 GB L4. PEFT's `model.disable_adapter()` is a context manager that routes the forward pass through the base weights for the duration of that block, then restores LoRA-enabled forwarding when the block exits. Net effect: one model in VRAM, two behaviours — and one shared model load (~5 min) instead of two.

## § 0 — Config
> Edit this cell only.

In [ ]:
import os, sys, subprocess, time, warnings, json, random
import numpy as np

# ── Repo ─────────────────────────────────────────────────────────────────
GITHUB_REPO = "https://github.com/antonisbast/eclipse-thesis.git"
REPO_DIR    = "/content/eclipse-thesis"

# ── Model + adapter ──────────────────────────────────────────────────────
MODEL_ID:       str  = "unsloth/gemma-4-E4B-it"
LOAD_IN_4BIT:   bool = True
MAX_SEQ_LEN:    int  = 1024
MAX_NEW_TOKENS: int  = 200

ADAPTER_PATH = "/content/drive/MyDrive/eclipse-thesis/sft_adaptersV6/lora_adapter"
ADAPTER_TAG  = "v6"   # label suffix in result tables

CITYLEARN_VERSION = "2.6.0b2"

# ── SAC expert — the distillation teacher — loaded from Drive ─────────────
# The SFT adapter was distilled from this SAC agent, so SAC is the natural
# reference: "how close to the teacher did distillation get?". Place the
# trained sac_*.pkl in SAC_DRIVE_DIR. Set USE_SAC=False to skip it.
USE_SAC:       bool = True
SAC_DRIVE_DIR: str  = "/content/drive/MyDrive/eclipse-thesis/agents"

# ── Seasonal panel — generalisation test on UNSEEN buildings ─────────────
# The SFT adapter was trained on buildings 0-5. Every config below uses
# buildings 6,7,8 — indices the model has NEVER seen — so the panel measures
# generalisation. 4 windows x 7 whole days, one per season; no single-week run.
EVAL_LEN: int = 168
UNSEEN_TRIPLE = [6, 7, 8]
CONFIGS = [
    {"name": "W1", "start": 1440, "buildings": UNSEEN_TRIPLE},
    {"name": "W2", "start": 3624, "buildings": UNSEEN_TRIPLE},
    {"name": "W3", "start": 5808, "buildings": UNSEEN_TRIPLE},
    {"name": "W4", "start": 7992, "buildings": UNSEEN_TRIPLE},
]

# ── Misc ─────────────────────────────────────────────────────────────────
HF_TOKEN: str = ""
SEED          = 42

import torch
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f"GPU: {g.name}  ({g.total_memory / 1e9:.1f} GB VRAM)")
else:
    print("No GPU — Runtime -> Change runtime type -> T4/L4")

_n_slm = 2 * len(CONFIGS)
print(f"Panel : {len(CONFIGS)} windows x (No-Control + RBC + SAC + Base SLM + SFT SLM)")
print(f"        buildings {UNSEEN_TRIPLE} (unseen)  |  {EVAL_LEN} steps/window")
print(f"        {_n_slm} SLM rollouts ~ {_n_slm * 60} min wall on L4 — § 6 is resumable,")
print(f"        run it config by config across sessions.")

## § 1 — Installs (CityLearn 2.6.0b2 + Unsloth Gemma-4 stack)

CityLearn pinned to **2.6.0b2** (`src.eval` uses `evaluate_v2()`), then the
**Unsloth Gemma-4 stack** — the *same* install as nb 05, with `transformers`
pinned to `5.5.0`. Loading the base Gemma through Unsloth's `FastModel` (rather
than plain `transformers` 4-bit) means this eval runs on the exact patched
kernels nb 05 trained the LoRA adapter against — no train/eval kernel drift.

In [ ]:
# CityLearn 2.6 is a pre-release on PyPI — pin the version explicitly.
# --no-deps because CityLearn pulls heavy/legacy deps we don't need at eval
# time (e.g. some EnergyPlus extras). We install the runtime deps explicitly.
!pip install -q numpy gymnasium doe-xstock nrel-pysam
!pip install -q --pre "CityLearn=={CITYLEARN_VERSION}" --no-deps

import citylearn
assert citylearn.__version__.startswith("2.6"), (
    f"Expected CityLearn 2.6.x, got {citylearn.__version__}. "
    f"src.eval depends on evaluate_v2() which only exists in 2.6+."
)
print(f"✓ CityLearn {citylearn.__version__}")

In [ ]:
# Unsloth + Gemma-4 stack — SAME install as nb 05, so the base Gemma here is
# loaded on the exact Unsloth-patched kernels nb 05 trained against. Loading
# with plain AutoModelForCausalLM (the old path) would eval the SFT adapter on
# different attention/RoPE code than it was trained on.
# transformers is PINNED to 5.5.0 — unsloth-zoo hard-caps it at <=5.5.0; an
# unpinned `--upgrade` pulls 5.8.x and the Gemma-4 patch breaks.
import os, re, torch

_v = re.match(r"[\d]+\.[\d]+", str(torch.__version__)).group(0)
_xformers = "xformers==" + {"2.10": "0.0.34", "2.9": "0.0.33.post1",
                            "2.8": "0.0.32.post2"}.get(_v, "0.0.34")

!pip install -q sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install -q --no-deps unsloth_zoo bitsandbytes accelerate {_xformers} peft trl triton unsloth
!pip install -q --no-deps --upgrade "torchao>=0.16.0"
!pip install -q --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install -q torchcodec
!pip install -q --no-deps --upgrade timm

torch._dynamo.config.recompile_limit = 64

import unsloth                       # triggers Unsloth's model patching
import transformers
print(f"✓ ML dependencies installed  |  transformers=={transformers.__version__}")
assert transformers.__version__.startswith("5.5"), (
    f"transformers is {transformers.__version__}, expected 5.5.x — re-run this cell."
)

## § 2 — Clone repo + mount Drive

In [ ]:
if not os.path.exists(REPO_DIR):
    res = subprocess.run(["git", "clone", GITHUB_REPO, REPO_DIR], capture_output=True, text=True)
    print(res.stdout or res.stderr)
else:
    res = subprocess.run(["git", "-C", REPO_DIR, "pull"], capture_output=True, text=True)
    print("Repo present —", res.stdout.strip() or "up to date")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from google.colab import drive
drive.mount('/content/drive')

assert os.path.isdir(ADAPTER_PATH), (
    f"Adapter not found at {ADAPTER_PATH}.\n"
    f"Inspect candidates:  !ls /content/drive/MyDrive/eclipse-thesis/sft_adaptersV5/"
)
print(f"✓ Adapter found at {ADAPTER_PATH}")

## § 3 — Load Base Gemma (Unsloth `FastModel`, 4-bit) + Attach the Saved LoRA

`FastModel.from_pretrained` pulls the base Gemma from HF Hub in `bitsandbytes` 4-bit and patches in Unsloth's kernels — the **same path nb 05 used to train the adapter**, so the eval and the training run share identical attention / RoPE code. `get_chat_template(tokenizer, "gemma-4")` then pins the inference chat template to exactly the one training rendered.

Once the base is in VRAM, `PeftModel.from_pretrained(base, ADAPTER_PATH)` attaches the nb 05 LoRA **in place** on the same object. The resulting `model` is a `PeftModel`: its `disable_adapter()` context manager routes the forward through the base weights for the duration of that block, then restores LoRA-enabled forwarding. Net effect: one model in VRAM (~9.5 GB for Gemma-4 E4B in 4-bit), two behaviours, one shared load.

In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
from peft import PeftModel
import torch, time

torch.cuda.empty_cache()

# Load the base Gemma via Unsloth FastModel — the SAME patched kernels nb 05
# trained the adapter against. (Loading with plain AutoModelForCausalLM would
# run the SFT adapter on different attention/RoPE code than training used.)
_t0 = time.time()
model, tokenizer = FastModel.from_pretrained(
    model_name      = MODEL_ID,
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = None,            # Unsloth auto-detects precision
    load_in_4bit    = LOAD_IN_4BIT,
    full_finetuning = False,
    token           = HF_TOKEN or None,
)
print(f"Base model loaded in {time.time()-_t0:.1f}s")

# Render inference with the exact chat template nb 05 trained against.
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

# Attach the nb 05 LoRA in-place. The result is a PeftModel — its
# disable_adapter() context manager routes the forward through the base
# weights, which is what the base-vs-SFT toggle in § 4 relies on.
_t0 = time.time()
model = PeftModel.from_pretrained(model, ADAPTER_PATH)
model.eval()
print(f"LoRA adapter attached in {time.time()-_t0:.1f}s")

# Guard: a missing disable_adapter() would silently make the "base" rollout
# identical to the SFT rollout — fail loud instead.
assert hasattr(model, "disable_adapter"), (
    "Loaded model has no disable_adapter() — the base-vs-SFT toggle in § 4 "
    "needs it. Check the peft / Unsloth versions installed in § 1."
)
print(f"✓ Ready — GPU mem: {torch.cuda.memory_allocated()/1e9:.1f} GB")

## § 4 — Build Base + SFT Providers (`src.providers.LocalHFProvider`)

Both providers wrap the **same** in-memory model object — only their constructor flags differ. The unified `LocalHFProvider` (the same class nb 03 uses for zero-shot SLMs and nb 05 uses for the post-SFT eval) takes `model=` / `tokenizer=` to skip re-loading and `disable_adapter=True` to route the forward through the base weights.

- `slm_sft`  — adapter enabled, prompt = `make_sft_prompt` (no-CoT, exactly what the model was trained on).
- `slm_base` — `disable_adapter=True`, prompt = `make_minimal_prompt` (CoT, the canonical zero-shot prompt the base Gemma has never been fine-tuned to ignore).

Picking the prompt that matches each model's training distribution is critical — using the wrong one is OOD and degrades sharply (see nb 05 § 19).

In [ ]:
from src.providers import LocalHFProvider
from src.sft       import make_sft_prompt
from src.agent     import make_minimal_prompt

# SFT provider: adapter enabled, no-CoT prompt (matches SFT training).
slm_sft = LocalHFProvider(
    model           = model,
    tokenizer       = tokenizer,
    model_id        = MODEL_ID,
    max_new_tokens  = MAX_NEW_TOKENS,
    prompt_builder  = make_sft_prompt,
    label           = f"sft_{ADAPTER_TAG}:{MODEL_ID.split('/')[-1]}",
)

# Base provider: SAME model object, but disable_adapter=True so every forward
# bypasses the LoRA → pure base Gemma. CoT prompt is the canonical zero-shot
# prompt (the base model has never seen the no-CoT SFT prompt).
slm_base = LocalHFProvider(
    model           = model,
    tokenizer       = tokenizer,
    model_id        = MODEL_ID,
    max_new_tokens  = MAX_NEW_TOKENS,
    prompt_builder  = make_minimal_prompt,
    disable_adapter = True,
    label           = f"base:{MODEL_ID.split('/')[-1]}",
)

# Ordered dict — table columns and plot panels follow this order downstream.
SLM_PROVIDERS = {"base": slm_base, "sft": slm_sft}
for tag, p in SLM_PROVIDERS.items():
    print(f"  [{tag:>4s}] {p.label}  (prompt={p.prompt_builder.__name__}, disable_adapter={p.disable_adapter})")

## § 5 — Env Factory + Rollout Helpers

`make_eval_env` builds a CityLearn env via the **auto-download schema** (robust on a fresh Colab kernel — no local dataset needed). One rollout helper per policy: no-op, RBC, the **SAC expert** (transfer baseline — its 6 trained policies applied positionally to the 3 unseen buildings), and the SLM (base or SFT, selected by the provider). The SAC pickle is loaded here from Drive.

In [ ]:
import re, glob, pickle, collections
from citylearn.citylearn  import CityLearnEnv
from citylearn.agents.rbc import BasicBatteryRBC
import citylearn

from src.env  import OBSERVATIONS, ACTIVE_ACTIONS, MERLINReward, snapshot_state
from src.sft  import render_state
from src.eval import evaluate, comparison_table

warnings.filterwarnings("ignore")
np.random.seed(SEED)
random.seed(SEED)


def make_eval_env(start: int, length: int, buildings: list) -> CityLearnEnv:
    """CityLearn env via the auto-download schema — robust on fresh Colab."""
    env = CityLearnEnv(
        schema="citylearn_challenge_2022_phase_all",
        buildings=buildings,
        central_agent=False,
        active_actions=ACTIVE_ACTIONS,
        active_observations=OBSERVATIONS,
        random_seed=SEED,
        simulation_start_time_step=start,
        simulation_end_time_step=start + length - 1,
    )
    env.reward_function = MERLINReward(env.get_metadata())
    return env


# Extracts CHARGE_X / IDLE / DISCHARGE_X tokens from the raw SLM response.
_TOK_RE = re.compile(r"<action building=\d+>([A-Z_0-9]+)</action>")


def run_slm_rollout(cfg: dict, provider, tag: str) -> dict:
    """One SLM rollout for one config x one provider (base or SFT)."""
    buildings = cfg["buildings"]
    n_b       = len(buildings)
    env       = make_eval_env(cfg["start"], EVAL_LEN, buildings)
    env.reset()
    tokens = collections.Counter()
    n_fb, t, t0, done = 0, 0, time.time(), False
    while not done:
        snap = snapshot_state(env)
        acts, raw, fb = provider.step(render_state(snap), n_buildings=n_b)
        n_fb += int(fb)
        tokens.update(_TOK_RE.findall(raw))
        _, _, term, trunc, _ = env.step([[float(a)] for a in acts])
        done = bool(term or trunc)
        if t % 48 == 0:
            print(f"    [{cfg['name']}/{tag}] t={t:4d}/{EVAL_LEN}  fb={n_fb}  {time.time()-t0:.0f}s")
        t += 1
    print(f"    [{cfg['name']}/{tag}] done in {time.time()-t0:.0f}s | fallbacks={n_fb}/{t}")
    return {"env": env, "tokens": tokens, "fb": n_fb, "elapsed": time.time() - t0}


def run_rbc_rollout(cfg: dict) -> CityLearnEnv:
    env = make_eval_env(cfg["start"], EVAL_LEN, cfg["buildings"])
    BasicBatteryRBC(env=env).learn(episodes=1)
    return env


def run_noop_rollout(cfg: dict) -> CityLearnEnv:
    env  = make_eval_env(cfg["start"], EVAL_LEN, cfg["buildings"])
    env.reset()
    n_b, done = len(cfg["buildings"]), False
    while not done:
        _, _, term, trunc, _ = env.step([[0.0] for _ in range(n_b)])
        done = bool(term or trunc)
    return env


def run_sac_rollout(cfg: dict) -> CityLearnEnv:
    """SAC expert as a transfer baseline — 6 trained policies applied
    positionally to the 3 unseen buildings (uses policies 0,1,2)."""
    env    = make_eval_env(cfg["start"], EVAL_LEN, cfg["buildings"])
    obs, _ = env.reset()
    done   = False
    while not done:
        actions = sac_agent.get_post_exploration_prediction(obs, deterministic=True)
        obs, _, term, trunc, _ = env.step(actions)
        done = bool(term or trunc)
    return env


def district_net(env: CityLearnEnv) -> float:
    """Total district net electricity (kWh) over the episode — < 0 means export."""
    return float(sum(np.sum(b.net_electricity_consumption) for b in env.buildings))


# ── Load the SAC expert (distillation teacher) from Drive ─────────────────
sac_agent = None
if USE_SAC:
    try:
        _pk = sorted(glob.glob(f"{SAC_DRIVE_DIR}/sac_*.pkl"))
        assert _pk, f"No sac_*.pkl in {SAC_DRIVE_DIR}"
        with open(_pk[-1], "rb") as _f:
            sac_agent = pickle.load(_f)
        print(f"SAC expert : {os.path.basename(_pk[-1])}  "
              f"({len(sac_agent.action_names)} policies, device={sac_agent.device})")
    except Exception as _exc:
        print(f"[SAC] not loaded ({_exc}) — grid runs without the expert.")
        sac_agent = None

print(f"CityLearn {citylearn.__version__}  |  {len(CONFIGS)} configs queued "
      f"(buildings {UNSEEN_TRIPLE}, unseen)")

In [ ]:
import inspect
import textwrap
import transformers.processing_utils

# Dynamically patch the apply_chat_template bug in transformers
source = inspect.getsource(transformers.processing_utils.ProcessorMixin.apply_chat_template)
source = textwrap.dedent(source)  # Fix IndentationError from class method extraction
source = source.replace(
    'if content["type"] in ["image", "video"]',
    'if isinstance(content, dict) and content.get("type") in ["image", "video"]'
)

exec_globals = transformers.processing_utils.__dict__.copy()
exec(source, exec_globals)
transformers.processing_utils.ProcessorMixin.apply_chat_template = exec_globals['apply_chat_template']
print("✓ Patched transformers.processing_utils.ProcessorMixin.apply_chat_template")

## § 6 — Run the Panel Grid

Per config (one seasonal window): no-op + RBC + **SAC expert** + base SLM + SFT SLM. Each EvalResult is stored on the config.

**Resumable** — configs that already have results are skipped, and results are saved on the config object as each finishes. A Colab disconnect mid-grid only costs the unfinished window; just re-run this cell. Run §§ 7+ once the grid reports `4/4 configs complete`.

In [ ]:
import pandas as pd

POLICY_ORDER = ["No-Control", "RBC"]
if sac_agent is not None:
    POLICY_ORDER.append("SAC")
POLICY_ORDER += [p.label for p in SLM_PROVIDERS.values()]

for cfg in CONFIGS:
    if cfg.get("_results"):
        print(f"  {cfg['name']}: already done — skip")
        continue
    print(f"\n{'='*64}\n {cfg['name']}  start={cfg['start']}  "
          f"buildings={cfg['buildings']}  steps={EVAL_LEN}\n{'='*64}")

    print("  No-Control …"); env_no  = run_noop_rollout(cfg)
    print("  RBC …");        env_rbc = run_rbc_rollout(cfg)
    envs    = {"No-Control": env_no, "RBC": env_rbc}
    results = {"No-Control": evaluate(env_no,  label=f"No-Control@{cfg['name']}"),
               "RBC":        evaluate(env_rbc, label=f"RBC@{cfg['name']}")}

    if sac_agent is not None:
        print("  SAC …"); env_sac = run_sac_rollout(cfg)
        envs["SAC"]    = env_sac
        results["SAC"] = evaluate(env_sac, label=f"SAC@{cfg['name']}")

    tokens, fb, t_sec = {}, {}, {}
    for tag, provider in SLM_PROVIDERS.items():
        print(f"  SLM/{tag} ({provider.label}) …")
        out = run_slm_rollout(cfg, provider, tag)
        envs[provider.label]    = out["env"]
        results[provider.label] = evaluate(out["env"], label=f"{provider.label}@{cfg['name']}")
        tokens[tag], fb[tag], t_sec[tag] = out["tokens"], out["fb"], out["elapsed"]

    cfg["_envs"]     = envs
    cfg["_results"]  = results
    cfg["_tokens"]   = tokens
    cfg["_fb"]       = fb
    cfg["_t_sec"]    = t_sec
    cfg["_noop_net"] = district_net(env_no)
    print(f"  regime: no-control net={cfg['_noop_net']:+.1f} kWh  -> "
          f"{'import — cost KPI valid' if cfg['_noop_net'] > 0 else 'EXPORT — cost KPI degenerate'}")
    for tag, toks in cfg["_tokens"].items():
        print(f"  action tokens ({tag}): {dict(toks)}")

_done = [c["name"] for c in CONFIGS if c.get("_results")]
print(f"\nGrid: {len(_done)}/{len(CONFIGS)} configs complete — {_done}")
if len(_done) == len(CONFIGS):
    total_min = sum(t for c in CONFIGS for t in c["_t_sec"].values()) / 60
    print(f"Total SLM wall time: {total_min:.1f} min")

## § 7 — Challenge & ZNE tables (via `src.eval.comparison_table`)
One Challenge-score table and one ZNE table per config. Indexed by agent
label, columns are the canonical KPIs the rest of the codebase uses.

In [ ]:
for cfg in CONFIGS:
    if not cfg.get("_results"):
        print(f"\n── {cfg['name']}: not run yet — skip (finish § 6 first)")
        continue
    valid = cfg["_noop_net"] > 0
    print(f"\n── {cfg['name']}  start={cfg['start']}  buildings={cfg['buildings']}  "
          f"[{'cost KPI valid' if valid else 'EXPORT — cost KPI DEGENERATE, read G / abs cost'}] ──")
    chal_df, zne_df = comparison_table(list(cfg["_results"].values()))
    print("Challenge scores (lower is better; 1.0 = no-control):")
    display(chal_df)
    print("ZNE / self-consumption:")
    display(zne_df[["ZNE ratio (solar / import)", "self-consumption ratio"]])

## § 7.1 — Cross-Config Summary (Phase I + ZNE per policy per config)

One row per (config × policy). The headline question for this notebook is **how does `sft` compare to `base` on the same unseen buildings?** — answered by the `Δ Phase I` delta at the bottom (negative = SFT beats base = distillation transferred to unseen buildings).

In [ ]:
KPI_KEYS = [("C  — cost", "C"), ("G  — carbon", "G"), ("R  — ramping", "R"),
            ("1L — load factor", "1L"), ("Phase I (C+G)/2", "Phase I"),
            ("Combined (C+G+D)/3", "Combined")]

rows = []
for cfg in CONFIGS:
    if not cfg.get("_results"):
        continue
    valid = cfg["_noop_net"] > 0
    for policy_lbl, res in cfg["_results"].items():
        rows.append({
            "config": cfg["name"], "C valid": "yes" if valid else "NO",
            "policy": policy_lbl,
            **{short: round(float(res.challenge[full]), 4) for full, short in KPI_KEYS},
            "ZNE ratio": round(float(res.zne["ZNE ratio (solar / import)"]), 4),
        })
summary_df = pd.DataFrame(rows).set_index(["config", "policy"])
print("Cross-config KPI summary — unseen buildings (generalisation):")
display(summary_df)

phase_i = summary_df["Phase I"].unstack("policy")   # used by § 8 spread

# ── Headline deltas — base vs SFT, and vs the expert ─────────────────────
base_label, sft_label = slm_base.label, slm_sft.label
_d = {"Δ Phase I (SFT − Base)": phase_i[sft_label] - phase_i[base_label],
      "Δ Phase I (SFT − RBC)":  phase_i[sft_label] - phase_i["RBC"]}
if "SAC" in phase_i.columns:
    _d["Δ Phase I (SFT − SAC)"] = phase_i[sft_label] - phase_i["SAC"]
deltas = pd.DataFrame(_d).round(4)
print("\nHeadline deltas — negative = first policy beats second:")
display(deltas)

# ── Base / SFT as a fraction of the SAC expert (valid-C configs only) ────
_valid = [c["name"] for c in CONFIGS if c.get("_results") and c["_noop_net"] > 0]
if "SAC" in phase_i.columns and _valid:
    cV = summary_df["C"].unstack("policy")
    print(f"\nFraction of the SAC expert captured  (valid-C configs {_valid})")
    print("captured = (1 - policy) / (1 - SAC), per config, mean:\n")
    for lbl in (base_label, sft_label):
        line = f"  {lbl:32s}"
        for metric, frame in (("cost C", cV), ("Phase I", phase_i)):
            fr = [(1.0 - frame.loc[cn, lbl]) / (1.0 - frame.loc[cn, "SAC"])
                  for cn in _valid if (1.0 - frame.loc[cn, "SAC"]) > 0]
            line += f"   {metric}: {np.mean(fr)*100:6.1f}%" if fr else f"   {metric}:   n/a"
        print(line)
    print("\nPhase I is the primary thesis metric; the SLM validation gate is >=70% of SAC.")
else:
    print("\n(SAC not available, or no valid-C configs — capture-vs-SAC skipped.)")

## § 8 — Cross-Config Spread

`src.eval.generalisation_gap` was designed for an explicit (train-buildings, unseen-buildings) pair. In this notebook *every* config is on unseen buildings, so the relevant comparison is the spread of each policy's score across configs — how stable is the SFT model when only the season / building subset changes? A tight spread on `sft` and a wide spread on `base` would mean SFT made the policy more robust.

In [ ]:
if len(CONFIGS) < 2:
    print("Need ≥ 2 configs in the eval grid to measure spread.")
else:
    spread = pd.DataFrame({
        "Phase I (mean)": phase_i.mean(axis=0),
        "Phase I (std)":  phase_i.std (axis=0),
        "Phase I (min)":  phase_i.min (axis=0),
        "Phase I (max)":  phase_i.max (axis=0),
    }).round(4)
    print("Per-policy Phase-I spread across configs (lower mean = better, lower std = more robust):")
    display(spread)

## § 8.1 — Panel Calibration

Does the 4-window panel proxy a full year? SAC is free to run, so we score it on the whole year (same unseen buildings) and compare to its panel mean over the valid-C windows. A small gap means the panel is a trustworthy cheap proxy — and only one expensive full-year SLM run is ever needed.

In [ ]:
if sac_agent is None:
    print("SAC not loaded — calibration skipped (set USE_SAC=True, provide the pickle).")
else:
    print("Running SAC full-year on the unseen buildings …")
    _env_yr = make_eval_env(0, 8760, UNSEEN_TRIPLE)
    _obs, _ = _env_yr.reset()
    _done = False
    while not _done:
        _a = sac_agent.get_post_exploration_prediction(_obs, deterministic=True)
        _obs, _, _term, _trunc, _ = _env_yr.step(_a)
        _done = bool(_term or _trunc)
    _c_year = float(evaluate(_env_yr, "SAC full-year").challenge["C  — cost"])

    _cV    = summary_df["C"].unstack("policy")
    _valid = [c["name"] for c in CONFIGS if c.get("_results") and c["_noop_net"] > 0]
    print(f"\nSAC full-year C (buildings {UNSEEN_TRIPLE})        : {_c_year:.4f}")
    if "SAC" in _cV.columns and _valid:
        _c_panel = float(np.mean([_cV.loc[cn, "SAC"] for cn in _valid]))
        print(f"SAC panel-mean C (valid configs {_valid}) : {_c_panel:.4f}")
        print(f"Proxy error |panel - year|                  : {abs(_c_panel - _c_year):.4f}")
        print("\n-> a small error confirms the valid-C panel windows proxy the full year.")
    else:
        print("(no valid-C configs to calibrate against)")

## § 9 — Action Distribution per (Config × Provider)

The base Gemma and SFT Gemma should produce visibly different action mixes. The SFT model was distilled from SAC, whose policy on this dataset is heavily biased toward `DISCHARGE_20` / `IDLE` / `CHARGE_20` (small actions dominate the optimum). A `base` row that's mostly `CHARGE_100` / `DISCHARGE_100` or stuck on `IDLE` is the expected zero-shot behaviour; the `sft` row should look much more like the teacher's distribution.

In [ ]:
TOKEN_ORDER = ["CHARGE_100","CHARGE_80","CHARGE_60","CHARGE_40","CHARGE_20",
               "IDLE",
               "DISCHARGE_20","DISCHARGE_40","DISCHARGE_60","DISCHARGE_80","DISCHARGE_100"]

rows = []
for cfg in CONFIGS:
    for tag in SLM_PROVIDERS:
        tot = sum(cfg["_tokens"][tag].values()) or 1
        row = {"config": cfg["name"], "provider": tag}
        for tok in TOKEN_ORDER:
            row[tok] = round(100 * cfg["_tokens"][tag].get(tok, 0) / tot, 1)
        row["fallbacks"] = cfg["_fb"][tag]
        rows.append(row)

action_df = pd.DataFrame(rows).set_index(["config", "provider"])
print("Action-token distribution per (config × provider) — %:")
display(action_df)

summary_actions = pd.DataFrame({
    "CHARGE %":    action_df[[c for c in action_df.columns if c.startswith("CHARGE")]].sum(axis=1),
    "IDLE %":      action_df["IDLE"],
    "DISCHARGE %": action_df[[c for c in action_df.columns if c.startswith("DISCHARGE")]].sum(axis=1),
}).round(1)
print("\nCoarse charge / idle / discharge split:")
display(summary_actions)

## § 10 — Plots: Action Mix + SoC Traces

Two visualisations:

1. **Action-mix bar chart** — stacked CHARGE / IDLE / DISCHARGE percentages per (config × provider). Visually answers "did SFT change what the model does?".
2. **SoC traces** — one subplot per config, two panels per subplot (base vs SFT). Reveals whether either policy actually moves the battery — flat SoC means the agent is parking at IDLE and the reward gap is mostly RBC vs no-op, not SLM vs anything.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 4.5))
summary_actions.plot(kind="bar", stacked=True, ax=ax,
                     color=["#2ca02c", "#7f7f7f", "#d62728"])
ax.set_ylabel("% of action tokens")
ax.set_title("Action mix per (config × provider) — base vs SFT")
ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5))
ax.set_ylim(0, 100)
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# One row per config, one column per SLM provider (base | sft).
n_cfg = len(CONFIGS)
n_prov = len(SLM_PROVIDERS)
fig, axes = plt.subplots(nrows=n_cfg, ncols=n_prov,
                         figsize=(6 * n_prov, 2.6 * n_cfg),
                         sharex="col")
if n_cfg == 1:
    axes = np.array([axes])
if n_prov == 1:
    axes = axes.reshape(-1, 1)

for r, cfg in enumerate(CONFIGS):
    for c, (tag, provider) in enumerate(SLM_PROVIDERS.items()):
        ax  = axes[r, c]
        env = cfg["_envs"][provider.label]
        for i, b in enumerate(env.buildings):
            ax.plot(b.electrical_storage.soc, label=f"B{cfg['buildings'][i]}", lw=1.2)
        ax.set_ylim(-0.05, 1.05)
        ax.set_ylabel("SoC")
        ax.set_title(f"{cfg['name']}  |  {tag}  (start={cfg['start']})", fontsize=10)
        ax.legend(loc="upper right", ncol=len(cfg["buildings"]), fontsize=8)
        ax.grid(linestyle="--", alpha=0.4)

for ax in axes[-1, :]:
    ax.set_xlabel("Hour into eval window")

plt.tight_layout()
plt.show()

## § 11 — Persist to Drive

Save every artefact needed to remake the comparison plots without re-running the SLMs: the cross-config summary, the action-mix DataFrames (per-config × provider), the per-config Challenge / ZNE tables, and a manifest with model / adapter paths, eval window, wall-time per provider, and fallback counts. All filenames are timestamped under `eval_runs/{ADAPTER_TAG}_base_vs_sft_<stamp>/` so re-runs never overwrite an earlier experiment.

In [ ]:
from pathlib import Path

stamp   = time.strftime("%Y%m%d_%H%M%S")
out_dir = Path(f"/content/drive/MyDrive/eclipse-thesis/eval_runs/{ADAPTER_TAG}_base_vs_sft_{stamp}")
out_dir.mkdir(parents=True, exist_ok=True)

summary_df.to_csv     (out_dir / "summary_cross_config.csv")
action_df.to_csv      (out_dir / "action_distribution.csv")
summary_actions.to_csv(out_dir / "action_summary.csv")
deltas.to_csv         (out_dir / "headline_deltas.csv")
if len(CONFIGS) >= 2:
    spread.to_csv     (out_dir / "phase1_spread.csv")

# Per-config Challenge + ZNE tables (4 policies each).
for cfg in CONFIGS:
    chal_df, zne_df = comparison_table(list(cfg["_results"].values()))
    chal_df.to_csv(out_dir / f"challenge_{cfg['name']}.csv")
    zne_df.to_csv (out_dir / f"zne_{cfg['name']}.csv")

manifest = {
    "adapter_path":      ADAPTER_PATH,
    "adapter_tag":       ADAPTER_TAG,
    "model_id":          MODEL_ID,
    "citylearn_version": citylearn.__version__,
    "eval_len":          EVAL_LEN,
    "providers":         {tag: p.label for tag, p in SLM_PROVIDERS.items()},
    "configs":           [{k: v for k, v in c.items() if not k.startswith('_')}
                          for c in CONFIGS],
    "slm_wall_time_sec": {c["name"]: c["_t_sec"] for c in CONFIGS},
    "fallbacks":         {c["name"]: c["_fb"]    for c in CONFIGS},
}
(out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(f"✓ Saved to {out_dir}")

## § 12 — Findings & Reading Guide

### Why a seasonal panel, not a single config

1. **The cost KPI degenerates in solar-rich windows.** `C` is a ratio — controlled cost / no-control cost. Where solar ≈ load the no-control district is roughly net-zero, the baseline cost collapses toward zero, and the ratio explodes or flips sign. The grid flags each window's regime from the no-control net load (`C valid = NO` when the district net-exports); read `G` / absolute cost there instead. `G`, `R`, `1L` never degenerate.
2. **A single window is a biased estimator.** Performance is strongly seasonal. The 4-window panel averages over it; § 8.1 calibrates the panel against the full year.
3. **Compare like with like.** Base SLM, SFT SLM, SAC, and the baselines are all scored on the *same* unseen buildings and the *same* windows.

### What this notebook answers

- **Did SFT help on unseen buildings?** § 7.1 `Δ Phase I (SFT − Base)` — negative means the fine-tuned model generalises better than base Gemma.
- **Did SFT clear the rule-based floor?** `Δ Phase I (SFT − RBC)`.
- **How close to the teacher did distillation get?** SAC is the distillation teacher. `Δ Phase I (SFT − SAC)` and the *fraction-of-SAC-captured* line are the headline: the Phase 4 gate is **≥ 70% of SAC on Phase I**.

### How to read it

- Read **cost `C` and carbon `G` separately**. The nb 02 remote-API study found a model can tie SAC on cost yet reach only ~30% of SAC on Phase I because it is carbon-blind — the `G` column exposes that.
- The **§ 9 action-token distribution** and **§ 10 SoC traces** show *how* base vs SFT differ — a successful SFT shifts the action mix toward the SAC-like cycling pattern rather than "charge-and-park".
- Watch the **fallback counts**: the SFT model uses the no-CoT prompt; a high fallback rate means the adapter is not reliably emitting the action format on unseen buildings.